# Pipeline — run all tasks (Free Edition friendly)

Databricks **Free Edition** supports Jobs but with limits (5 concurrent tasks, daily quota). If creating a Workflow job is difficult, **run this notebook** instead — it chains `01`–`07` sequentially via `dbutils.notebook.run`.

**Widgets:** `pipeline_run_id`, `dry_run`, `simulate_failure` (passed to each child notebook).

In [ ]:
import sys, importlib, json, uuid
REPO_ROOT = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get().split("/notebooks/")[0]
if REPO_ROOT not in sys.path: sys.path.insert(0, REPO_ROOT)
import src.orchestration.params as op
importlib.reload(op)
from src.orchestration.params import bind_pipeline_widgets, get_pipeline_params

bind_pipeline_widgets(dbutils)
params = get_pipeline_params(dbutils)
run_id = params.pipeline_run_id or str(uuid.uuid4())
base = f"{REPO_ROOT}/notebooks/10_orchestration"
notebook_args = {
    "pipeline_run_id": run_id,
    "dry_run": "true" if params.dry_run else "false",
    "simulate_failure": params.simulate_failure or "",
}

TASKS = [
    ("bronze_ingestion", f"{base}/01_bronze_ingestion"),
    ("quality_checks", f"{base}/02_quality_checks"),
    ("silver_transforms", f"{base}/03_silver_transforms"),
    ("reconciliation", f"{base}/04_reconciliation"),
    ("gold_aggregations", f"{base}/05_gold_aggregations"),
    ("dimensional_refresh", f"{base}/06_dimensional_refresh"),
    ("visualization", f"{base}/07_visualization"),
]

results = []
for task_name, path in TASKS:
    entry = {"task": task_name, "notebook": path}
    try:
        print(f"Running {task_name}...")
        output = dbutils.notebook.run(path, timeout_seconds=7200, arguments=notebook_args)
        entry["status"] = "SUCCESS"
        if output:
            entry["output_preview"] = output[:500]
    except Exception as exc:
        entry["status"] = "FAILED"
        entry["error"] = str(exc)
        results.append(entry)
        print(json.dumps({"pipeline_run_id": run_id, "tasks": results}, indent=2))
        raise
    results.append(entry)

summary = {"pipeline_run_id": run_id, "tasks": results}
print(json.dumps(summary, indent=2))